### Retention / Churn Heuristic

For this initial baseline model, a player is considered **retained** if they recorded more than 60 minutes of playtime within the last two weeks. Players with 60 minutes or less are treated as likely churned users.

The 60-minute threshold acts as apractical behavioral heuristic rather than a definitive business rule. The assumption is that players who return and spend at least one hour actively engaging with a game over a recent two-week period demonstrate meaningful continued interest and engagement. In contrast, very low or zero recent playtime may indicate disengagement, abandonment, or temporary inactivity.

This threshold was intentionally chosen as a lightweight and interpretable starting point for experimentation. It helps transform continuous playtime behavior into a binary classification problem suitable for Logistic Regression while remaining easy to explain from a business perspective.

Future iterations of the project may refine this definition using:
- percentile-based engagement thresholds,
- genre-specific activity expectations,
- rolling activity windows,
- survival analysis,
- or clustering methods to identify natural retention breakpoints.

In [ ]:
# ===============================
# Check label distribution
# ===============================
# Quick Feat Engineering here
def feat_label(df):
    from pyspark.ml.feature import VectorAssembler
    from pyspark.sql import functions as F
    
    feature_cols = [
        "author_num_games_owned",
        "author_num_reviews",
        "author_playtime_forever",
        "author_playtime_last_two_weeks",
        "author_playtime_at_review",
        "author_last_played",
        "voted_up",
        "votes_up",
        "votes_funny",
        "weighted_vote_score",
        "comment_count",
        "written_during_early_access",
        "timestamp_created",
        "timestamp_updated",
    ]
    
    feat_df = (
        df
        .withColumn('churn', 
            F.when(F.col('author_playtime_last_two_weeks') > 60, 1)
            .otherwise(0)
        )
        .select(feature_cols + ['churn'])
        
    )

    final_df = (
        VectorAssembler(inputCols=feature_cols, outputCol='finalized_features')
        .transform(feat_df)
        .select('finalized_features', 'churn')
    )
        
    final_df.printSchema()
    memory_count(final_df, include_row_count=True)

    return final_df

train = train.transform(feat_label)
val = val.transform(feat_label)

train.groupBy('churn').count().show()

In [ ]:
# FUTURE DEVELOPMENT

# # text canonicalization
# lowercasing
# URL replacement
# emoji handling
# punctuation normalization
# accent normalization
# contraction handling
# tokenization
# stopword removal
# stemming / lemmatization
# language-specific processing

# # text vectorization & feature engineering
# TF-IDF
# word n-grams
# character n-grams
# sentiment scores
# embeddings
# language-specific vectorizers
# review length features